# Voynich context curves on a GPU

Small character-level transformers trained from scratch on 8 texts of 180,000 characters each (Voynich, two shuffled Voynich controls, Latin, Italian, Naibbe cipher, Rugg grille, autocopist), identical setup. Output: loss against context length per text (`summarize.py`), probes and attention reach (`probe.py`).

1. Runtime > Change runtime type > **T4 GPU**.
2. Run the cells in order. The upload cell asks for `voynich_tfm.zip`.
3. The last cell downloads `voynich_tfm_results.zip`; give it back to Claude.

In [ ]:
from google.colab import files
up = files.upload()   # choose voynich_tfm.zip
!unzip -o -q voynich_tfm.zip && ls voynich_tfm/tfm

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print(torch.__version__, torch.cuda.is_available())

## Settings
One place to change them. The calibration below must beat the Latin 5-gram (2.03 bits/char, in `tfm/results/ngram.json`); if it does not, raise `STEPS` or `D` and rerun it before the main loop.

In [ ]:
STEPS, LR, DROP, BATCH, D, LAYERS = 8000, 3e-3, 0.1, 64, 128, 4
SEEDS = [0, 1, 2]
TEXTS = ['voynich', 'latin', 'voynich_shufpage', 'voynich_shufline', 'naibbe', 'italian', 'rugg', 'autocopist']
ARGS = f'--steps {STEPS} --lr {LR} --drop {DROP} --batch {BATCH} --d {D} --layers {LAYERS}'
%cd /content/voynich_tfm

## Calibration (Latin, one seed)

In [ ]:
!python -W ignore tfm/train.py latin {ARGS} --tag calib_latin | grep -E 'step [0-9]*000 |RESULT'
import json; c = json.load(open('tfm/results/calib_latin.json')); ng = json.load(open('tfm/results/ngram.json'))
import numpy as np; m = float(np.mean(c['curve']))
print(f'model {m:.3f} bits/char vs 5-gram {ng["latin"]["5"]:.3f} ->', 'OK' if m < ng['latin']['5'] else 'UNDERTRAINED: raise STEPS or D')
!rm -f tfm/results/calib_latin.json tfm/models/calib_latin.pt

## Main loop: 8 texts x 3 seeds

In [ ]:
import subprocess
for seed in SEEDS:
    for t in TEXTS:
        out = subprocess.run(f'python -W ignore tfm/train.py {t} {ARGS} --seed {seed} --tag {t}_s{seed}', shell=True, capture_output=True, text=True).stdout
        print([l for l in out.splitlines() if l.startswith('RESULT')] or out[-500:], flush=True)

## Results

In [ ]:
!python tfm/summarize.py
!python -W ignore tfm/probe.py --model voynich_s0

In [ ]:
!zip -q -r voynich_tfm_results.zip tfm/results tfm/models
files.download('voynich_tfm_results.zip')